# Day 4 · 06 · Missing Data

Missing values are normal in real data. What matters is noticing them, understanding **why** they are missing, and choosing a treatment you can defend.

**In this notebook**
- What pandas treats as missing, and what it does not
- Profiling: how much is missing, and where
- How missing values behave in calculations
- `dropna`
- `fillna`: constants, statistics, group-wise values
- `ffill`, `bfill` and `interpolate` for ordered data
- Nullable dtypes: integers that can be missing
- Keeping a "was missing" flag
- Sentinel values that hide missing data

**Dataset:** `site_traffic.csv`, `orders_raw.csv`, `orders.csv`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../datasets").resolve()

orders = pd.read_csv(DATA / "orders.csv", parse_dates=["order_date"])
raw = pd.read_csv(DATA / "orders_raw.csv")
traffic = pd.read_csv(DATA / "site_traffic.csv", parse_dates=["timestamp"]).set_index("timestamp").sort_index()

## 1. What counts as missing?
pandas uses `NaN` (numbers), `NaT` (dates) and `pd.NA` (nullable dtypes). `isna()` detects all of them, and `None`.

Things that are **not** missing to pandas: an empty string `""`, a space `" "`, the text `"unknown"`, the number `0` or `-1`.

In [2]:
examples = pd.Series([1.5, np.nan, None, pd.NA, pd.NaT, "", " ", "unknown", 0], dtype="object")

pd.DataFrame({"value": examples.map(repr), "isna": examples.isna()})

,value,isna
0,1.5,False
1,nan,True
2,None,True
3,<NA>,True
4,NaT,True
5,'',False
6,' ',False
7,'unknown',False
8,0,False


### What `read_csv` converts for you
While reading, empty cells and a list of common markers (`NA`, `N/A`, `null`, `NaN`, ...) become missing. That is usually helpful, but it happens silently.

`orders_raw.csv` contains both blank quantities and the text `N/A`. Reading everything as text with `keep_default_na=False` shows what is really in the file.

In [3]:
as_text = pd.read_csv(DATA / "orders_raw.csv", dtype="str", keep_default_na=False)

print("in the file  : blank =", (as_text["quantity"] == "").sum(), "| 'N/A' =", (as_text["quantity"] == "N/A").sum())
print("default read : missing =", raw["quantity"].isna().sum(), "| dtype =", raw["quantity"].dtype)

in the file  : blank = 10 | 'N/A' = 9
default read : missing = 19 | dtype = float64


## 2. Profile the missing values
Start with counts and shares per column. `isna().mean()` is the share because `True` counts as 1.

In [4]:
def missing_report(frame):
    return pd.DataFrame(
        {
            "dtype": frame.dtypes.astype("str"),
            "missing": frame.isna().sum(),
            "missing_pct": frame.isna().mean().mul(100).round(1),
        }
    ).sort_values("missing", ascending=False)


missing_report(raw)

,dtype,missing,missing_pct
amount,str,23,5.7
email,str,20,5.0
quantity,float64,19,4.7
order_date,str,8,2.0
customer_id,str,5,1.2
order_id,int64,0,0.0
channel,str,0,0.0
status,str,0,0.0


In [5]:
# Per row: how many fields are missing in each record?
raw.isna().sum(axis=1).value_counts().sort_index().rename_axis("missing_fields").rename("rows")

missing_fields
0    333
1     62
2      5
3      1
Name: rows, dtype: int64

### Missing for a reason
`orders.staff_id` is empty for 1,596 of 2,000 orders. Before treating that as a problem, look at **where** it is missing: only store orders have a sales rep. The gap is information, not an error.

In [6]:
orders["staff_id"].isna().groupby(orders["channel"]).agg(["sum", "mean"]).rename(columns={"sum": "missing", "mean": "share"})

,missing,share
channel,,
mobile,626,1.0
store,0,0.0
web,970,1.0


### Missing rows versus missing values
A time series can also be missing whole **rows**. `site_traffic.csv` has both problems. `asfreq("h")` inserts the absent hours so that every gap becomes a visible `NaN`.

In [7]:
hourly = traffic.asfreq("h")

pd.DataFrame(
    {
        "missing_values_as_loaded": traffic.isna().sum(),
        "after_adding_missing_rows": hourly.isna().sum(),
    }
)

,missing_values_as_loaded,after_adding_missing_rows
sessions,70,106
conversions,70,106
avg_load_ms,98,134


## 3. How missing values behave
- Aggregations (`sum`, `mean`, `min`, ...) **skip** missing values by default.
- Arithmetic with a missing value gives a missing result.
- `NaN == NaN` is `False`. Always test with `isna()`.
- `count()` counts non-missing values; `len()` and `size` count rows.

In [8]:
sessions = hourly["sessions"]

print("rows                 :", len(sessions))
print("count (non-missing)  :", sessions.count())
print("mean, missing skipped:", round(sessions.mean(), 2))
print("sum with skipna=False:", sessions.sum(skipna=False))
print("NaN == NaN           :", np.nan == np.nan)
print("rows where sessions == NaN:", (sessions == np.nan).sum(), "| with isna():", sessions.isna().sum())

rows                 : 2160
count (non-missing)  : 2054
mean, missing skipped: 174.86
sum with skipna=False: nan
NaN == NaN           : False
rows where sessions == NaN: 0 | with isna(): 106


In [9]:
# A sum of all-missing values is 0 unless you ask for a minimum number of observations.
outage_hours = sessions.loc["2025-02-10 03:00":"2025-02-10 14:00"]

print("values present:", outage_hours.count())
print("sum()            :", outage_hours.sum())
print("sum(min_count=1) :", outage_hours.sum(min_count=1))

values present: 0
sum()            : 0.0
sum(min_count=1) : nan


## 4. `dropna`: remove rows or columns
| Call | Removes |
| --- | --- |
| `dropna()` | rows with **any** missing value |
| `dropna(subset=["a", "b"])` | rows where `a` or `b` is missing |
| `dropna(how="all")` | rows where **everything** is missing |
| `dropna(thresh=n)` | rows with fewer than `n` non-missing values |

Dropping is safe when few rows are affected and they are missing at random. It becomes biased when missingness is related to the value, for example when failed payments have no amount.

In [10]:
print("all rows                          :", len(raw))
print("dropna()                          :", len(raw.dropna()))
print("dropna(subset=['amount'])         :", len(raw.dropna(subset=["amount"])))
print("dropna(subset=['customer_id', 'order_date']):", len(raw.dropna(subset=["customer_id", "order_date"])))
print("dropna(thresh=7)                  :", len(raw.dropna(thresh=7)))

all rows                          : 401
dropna()                          : 333
dropna(subset=['amount'])         : 378
dropna(subset=['customer_id', 'order_date']): 388
dropna(thresh=7)                  : 395


## 5. `fillna`: replace with a value
- A constant: `fillna(0)`, `fillna("unknown")`
- A dictionary: a different value per column
- A statistic: median or mean
- A **group-wise** statistic through `transform`, which respects differences between groups

`0` is only correct when missing really means zero.

In [11]:
filled = raw.fillna({"email": "unknown", "quantity": raw["quantity"].median()})
print(filled[["email", "quantity"]].isna().sum().to_dict())

{'email': 0, 'quantity': 0}


In [12]:
# Page-load time differs by hour of day, so fill each gap with the median for that hour.
load = hourly["avg_load_ms"]
hour_median = load.groupby(load.index.hour).transform("median")

comparison = pd.DataFrame(
    {
        "original": load,
        "overall_median": load.fillna(load.median()),
        "median_for_that_hour": load.fillna(hour_median),
    }
)
comparison[load.isna()].round(1).head(4)

,original,overall_median,median_for_that_hour
timestamp,,,
2025-01-01 14:00:00,NaN,419.6,416.4
2025-01-01 19:00:00,NaN,419.6,422.2
2025-01-02 11:00:00,NaN,419.6,420.7
2025-01-02 14:00:00,NaN,419.6,416.4


## 6. Ordered data: `ffill`, `bfill` and `interpolate`
When rows have an order, neighbours carry information.

| Method | Fills with |
| --- | --- |
| `ffill()` | the last known value |
| `bfill()` | the next known value |
| `interpolate()` | values on a line between the neighbours |

Use `limit=` to cap how many consecutive gaps are filled. Without it, a long outage is papered over with invented data.

In [13]:
window = hourly.loc["2025-03-05 00:00":"2025-03-05 07:00", "sessions"]

pd.DataFrame(
    {
        "original": window,
        "ffill": window.ffill(),
        "ffill(limit=2)": window.ffill(limit=2),
        "interpolate": window.interpolate(method="time").round(1),
    }
)

,original,ffill,ffill(limit=2),interpolate
timestamp,,,,
2025-03-05 00:00:00,120.0,120.0,120.0,120.0
2025-03-05 01:00:00,NaN,120.0,120.0,118.0
2025-03-05 02:00:00,NaN,120.0,120.0,116.0
2025-03-05 03:00:00,NaN,120.0,NaN,114.0
2025-03-05 04:00:00,NaN,120.0,NaN,112.0
2025-03-05 05:00:00,NaN,120.0,NaN,110.0
2025-03-05 06:00:00,NaN,120.0,NaN,108.0
2025-03-05 07:00:00,106.0,106.0,106.0,106.0


In [14]:
# limit=3 fills at most 3 consecutive hours of each gap.
repaired = sessions.interpolate(method="time", limit=3, limit_area="inside")

print("missing before:", sessions.isna().sum())
print("missing after :", repaired.isna().sum())

missing before: 106
missing after : 12


`limit=3` fills the first three hours of **every** gap, including the start of the long ones. To fill only gaps that are short in total, measure each gap's length first.

In [15]:
is_missing = sessions.isna()
gap_id = (is_missing != is_missing.shift()).cumsum()
gap_length = is_missing.groupby(gap_id).transform("sum")

short_gap = is_missing & (gap_length <= 3)
careful = sessions.where(~short_gap, sessions.interpolate(method="time"))

print("missing hours in short gaps (filled):", int(short_gap.sum()))
print("missing hours in long gaps (kept)   :", int(careful.isna().sum()))

missing hours in short gaps (filled): 88
missing hours in long gaps (kept)   : 18


## 7. Nullable dtypes
A classic NumPy integer column cannot hold a missing value, so one `NaN` turns the whole column into **floats**: `3` becomes `3.0`.

The nullable dtypes (`Int64`, `Float64`, `boolean`, with a capital letter) keep the type and use `pd.NA` for the gaps. `convert_dtypes()` picks them for every column.

In [16]:
quantity = raw["quantity"]
print("default dtype:", quantity.dtype, "->", quantity.head(3).tolist())

nullable = quantity.astype("Int64")
print("nullable     :", nullable.dtype, "->", nullable.head(3).tolist())
print("still missing:", nullable.isna().sum(), "| sum:", nullable.sum())

raw.convert_dtypes().dtypes

default dtype: float64 -> [4.0, 5.0, 6.0]
nullable     : Int64 -> [4, 5, 6]
still missing: 19 | sum: 1573


order_id        Int64
customer_id    string
email          string
order_date     string
status         string
channel        string
quantity        Int64
amount         string
dtype: object

In [17]:
# Turning text into numbers: errors="coerce" converts what it can and marks the rest as missing.
amount_text = raw["amount"].str.replace("$", "", regex=False).str.replace(",", "", regex=False)
amount = pd.to_numeric(amount_text, errors="coerce")

print("missing in the raw text:", raw["amount"].isna().sum(), "| after conversion:", amount.isna().sum())

missing in the raw text: 23 | after conversion: 23


Compare the count of missing values before and after a conversion. If it rises, `coerce` hid some values that could not be parsed.

## 8. Keep a flag
Once a value is filled, nobody can tell it was missing. Store that fact in a boolean column **before** filling. Later analysis can exclude or separately examine the filled rows.

In [18]:
flagged = hourly[["sessions"]].copy()
flagged["sessions_was_missing"] = flagged["sessions"].isna()
flagged["sessions"] = flagged["sessions"].interpolate(method="time", limit=3)

flagged.loc["2025-03-05 00:00":"2025-03-05 04:00"].round(1)

,sessions,sessions_was_missing
timestamp,,
2025-03-05 00:00:00,120.0,False
2025-03-05 01:00:00,118.0,True
2025-03-05 02:00:00,116.0,True
2025-03-05 03:00:00,114.0,True
2025-03-05 04:00:00,NaN,True


## 9. Sentinels: missing values in disguise
Some systems write a placeholder instead of leaving a field empty: `0`, `-1`, `9999`, `"N/A"`, `"unknown"`. pandas sees ordinary values, so they flow into your averages.

In `orders_raw.csv` a quantity of `0` or `-1` cannot be a real order, and customer `C9999` does not exist. Convert such placeholders to real missing values.

In [19]:
print("quantity values that cannot be real:", raw.loc[raw["quantity"] <= 0, "quantity"].value_counts().to_dict())
print("mean quantity as loaded            :", round(raw["quantity"].mean(), 3))

cleaned_quantity = raw["quantity"].where(raw["quantity"] > 0)
print("mean with placeholders as missing  :", round(cleaned_quantity.mean(), 3))

customer = raw["customer_id"].replace({"C9999": pd.NA})
print("missing customer ids: before", raw["customer_id"].isna().sum(), "| after", customer.isna().sum())

quantity values that cannot be real: {0.0: 11, -1.0: 4}
mean quantity as loaded            : 4.118
mean with placeholders as missing  : 4.297
missing customer ids: before 5 | after 17


## Choosing a treatment
| Situation | Reasonable treatment |
| --- | --- |
| Field is required and few rows lack it | drop the rows, and report how many |
| Missing means "none" | fill with `0` or `"none"` |
| Numeric, roughly random gaps | median, ideally per group |
| Ordered data, short gaps | `ffill` or `interpolate` with a limit |
| Long gaps | leave missing and exclude the period |
| Missingness itself is informative | keep a flag column |

## Key takeaways
- Check `isna().sum()` on every new dataset, and check for missing **rows** in time series.
- Ask why values are missing before choosing a fix.
- Aggregations skip missing values; totals over incomplete data mislead.
- Limit how far `ffill` and `interpolate` may reach.
- Use nullable dtypes to keep integers as integers.
- Hunt for sentinel values.

## Practice
1. Report the share of missing `avg_load_ms` per week.
2. Fill missing `conversions` using the average conversion rate of that hour of day times `sessions`.
3. Find the longest run of consecutive missing `sessions` after `asfreq`.
4. In `raw`, how many rows have a missing `amount` **and** a missing `quantity`?